# Curvilinear Boundary Conditions

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook connects ghost zones, parity signs, one-sided radiation
stencils, and generated BHaH boundary routines for curvilinear grids.

Navigation: [Index][index] | Previous: [Curvilinear Wave Equation][prev]
| Next: [GeneralRFM and Fisheye Coordinates][next]

[index]: ../index.ipynb
[prev]: ../3-wave_equation/wave_equation_curvilinear.ipynb
[next]: generalrfm_and_fisheye.ipynb

### Required Reading

- [Boundary Conditions and Convergence][bc-convergence]
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)

[bc-convergence]: ../2-numerical_methods/boundary_conditions_and_convergence.ipynb

### NRPy Source Code

- [Boundary metadata setup][b]: builds parity and in-domain maps.
- [Outer radiation routines][o]: builds one-sided radiation support.
- [Inner-only routine][i]: registers the generated inner fill.
- [Selected-gridfunction inner routine][s]: registers
  selected-gridfunction inner fills.
- [Outer extrapolation routine][x]: registers extrapolated
  outer fills with inner fills.
- [Boundary registration entry point][r]: registers all boundary
  routines used by generated projects.

[b]:https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/CurviBoundaryConditions/bcstruct_set_up.py
[o]:https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/CurviBoundaryConditions/apply_bcs_outerradiation_and_inner.py
[i]:https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/CurviBoundaryConditions/apply_bcs_inner_only.py
[s]:https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/CurviBoundaryConditions/apply_bcs_inner_only_specific_gfs.py
[x]:https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/CurviBoundaryConditions/apply_bcs_outerextrap_and_inner.py
[r]:https://github.com/nrpy/nrpy/blob/main/nrpy/infrastructures/BHaH/CurviBoundaryConditions/register_all.py

# Table of Contents

1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Boundary and C-Function Structure](#Boundary-and-C-Function-Structure)
1. [Step 1](#Step-1:-Import-Boundary-Condition-Tools): Import tools.
1. [Step 2](#Step-2:-Connect-Parity-Classes-to-Boundary-Fills): Catalog
   parity classes.
1. [Step 3](#Step-3:-Generate-Spherical-Parity-Assignments): Generate parity.
1. [Validation Check](#Validation-Check): Validate parity signs.
1. [Step 4](#Step-4:-Check-the-Outer-Radiation-Stencil): Validate stencil.
1. [Step 5](#Step-5:-Inspect-Generated-Boundary-Routines): Inspect C functions.
1. [Learning Check](#Learning-Check)
1. [Continue](#Continue)

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **Ghost zone:** extra grid points outside the physical domain, filled from
  boundary rules.
- **Inner boundary:** a coordinate boundary, such as the spherical origin or
  axis, where a ghost-zone point can map back into the physical grid.
- **Outer boundary:** the edge of the physical domain where outgoing waves are
  allowed to leave.
- **In-domain point:** the physical-grid point that supplies data for a mapped
  ghost-zone point.
- **Parity sign:** the `+1` or `-1` multiplier applied when copying a tensor
  component from an in-domain point to a ghost-zone point.
- **Finite-difference stencil:** a weighted set of neighboring grid values
  used to approximate a derivative.
- **Finite-difference order:** the polynomial order through which the stencil
  gives the exact derivative.
- **Stencil width:** the number of grid points used by the derivative formula.
- **One-sided stencil:** a stencil that uses points mainly on one side of the
  target point.
- **BHaH:** the BlackHoles@Home infrastructure target used by generated NRPy
  C projects.
- **Gridfunction:** a named grid-stored field. Boundary routines write arrays
  of gridfunction values.
- **`gfs`:** the generated C pointer to gridfunction storage.
- **C function:** a named generated C routine with a return type, parameter
  list, and body.
- **Function prototype:** the C declaration showing the return type, name, and
  complete parameter list.
- **`CFunction_dict`:** the NRPy registry that stores generated C functions.
- **`commondata_struct`:** generated shared run metadata passed to BHaH C
  functions.
- **`params_struct`:** generated runtime parameters, grid sizes, and
  coordinate settings.
- **`bc_struct`:** generated boundary metadata, including ghost-zone maps and
  parity data.

# Boundary and C-Function Structure
### [Back to [top](#Table-of-Contents)]

The boundary-fill roadmap is:

1. Map an inner ghost-zone coordinate to an in-domain coordinate.
2. Copy the field value from the in-domain point.
3. Multiply by the component parity sign.
4. At the outer boundary, use one-sided in-domain data in an outgoing-wave rule.
5. Register generated C functions that apply these rules in a project.

For a component `A`, the inner-boundary copy has the form

$$\nu_A(q_{\rm ghost}) = P_A(q_{\rm ghost}, q_{\rm in})\nu_A(q_{\rm in}).$$

Here `P_A` is `+1` for scalar data and may be `-1` for vector or tensor
components whose basis directions flip. SENR/NRPy+ uses this reference-metric
and parity strategy for singular curvilinear coordinates; see
[SENR/NRPy+](https://arxiv.org/abs/1712.07658).

Generated C functions have this structure:

```c
/*
 * Description of what the function computes.
 */
return_type function_name(parameter_list) {
    local_declarations;
    assignments_or_algorithm;
}
```

In the generated metadata inspected below, `cfunc_type` is the return type,
`name` is the function name, `params` is the complete parameter list,
`includes` lists required headers, and `body` contains the generated algorithm.

# Step 1: Import Boundary-Condition Tools
### [Back to [top](#Table-of-Contents)]

This setup cell imports SymPy, the NRPy parameter interface, and the boundary
modules used below. It has no output because the next cell prints the first
catalog.

In [1]:
import sympy as sp

import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import (
    apply_bcs_outerradiation_and_inner as outer_bcs,
)
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import bcstruct_set_up

# Step 2: Connect Parity Classes to Boundary Fills
### [Back to [top](#Table-of-Contents)]

Each generated parity index describes which component sign can appear in the
inner-boundary copy formula. Scalars keep their sign. Vector and tensor signs
come from products of basis-direction signs.

Inspect the complete parity-class catalog:

- index `0` is scalar data;
- indices `1-3` are vector or one-form components;
- indices `4-9` are symmetric rank-2 tensor components.

In [2]:
parity_classes = [
    "scalar",
    "x0 vector or one-form component",
    "x1 vector or one-form component",
    "x2 vector or one-form component",
    "x0x0 symmetric rank-2 component",
    "x0x1 symmetric rank-2 component",
    "x0x2 symmetric rank-2 component",
    "x1x1 symmetric rank-2 component",
    "x1x2 symmetric rank-2 component",
    "x2x2 symmetric rank-2 component",
]
print("index | parity class | where used | inspect")
for index, name in enumerate(parity_classes):
    print(index, "|", name, "|", "inner ghost-zone fill", "|", "sign")

index | parity class | where used | inspect
0 | scalar | inner ghost-zone fill | sign
1 | x0 vector or one-form component | inner ghost-zone fill | sign
2 | x1 vector or one-form component | inner ghost-zone fill | sign
3 | x2 vector or one-form component | inner ghost-zone fill | sign
4 | x0x0 symmetric rank-2 component | inner ghost-zone fill | sign
5 | x0x1 symmetric rank-2 component | inner ghost-zone fill | sign
6 | x0x2 symmetric rank-2 component | inner ghost-zone fill | sign
7 | x1x1 symmetric rank-2 component | inner ghost-zone fill | sign
8 | x1x2 symmetric rank-2 component | inner ghost-zone fill | sign
9 | x2x2 symmetric rank-2 component | inner ghost-zone fill | sign


# Step 3: Generate Spherical Parity Assignments
### [Back to [top](#Table-of-Contents)]

The full generated assignment block maps each parity class to symbolic dot
products between basis directions at the ghost point and the in-domain point.

Inspect the complete generated block for:

- `REAL_parity_array[0] = 1`, because scalars keep their sign;
- entries `[1]` through `[3]`, because vector-like components can flip;
- entries `[4]` through `[9]`, because rank-2 signs are vector-sign products.

In [3]:
saved_boundary_params = {
    name: par.parval_from_str(name) for name in ["Infrastructure", "fp_type"]
}
try:
    par.set_parval_from_str("Infrastructure", "BHaH")
    par.set_parval_from_str("fp_type", "double")
    parity_code = bcstruct_set_up.parity_conditions_symbolic_dot_products("Spherical")
    assignment_block = parity_code[parity_code.index("{") :]
finally:
    for name, value in saved_boundary_params.items():
        par.set_parval_from_str(name, value)
print("complete Spherical parity assignment block:")
print(assignment_block)
print("restored runtime settings:", sorted(saved_boundary_params))

Setting up reference_metric[Spherical]...


complete Spherical parity assignment block:
{
const REAL tmp0 = cos(xx1)*cos(xx1_inbounds);
const REAL tmp1 = sin(xx1)*sin(xx1_inbounds);
const REAL tmp2 = sin(xx2)*sin(xx2_inbounds);
const REAL tmp3 = cos(xx2)*cos(xx2_inbounds);
const REAL tmp4 = tmp0 + tmp1*tmp2 + tmp1*tmp3;
const REAL tmp5 = tmp0*tmp2 + tmp0*tmp3 + tmp1;
const REAL tmp6 = tmp2 + tmp3;
REAL_parity_array[0] = 1;
REAL_parity_array[1] = tmp4;
REAL_parity_array[2] = tmp5;
REAL_parity_array[3] = tmp6;
REAL_parity_array[4] = ((tmp4)*(tmp4));
REAL_parity_array[5] = tmp4*tmp5;
REAL_parity_array[6] = tmp4*tmp6;
REAL_parity_array[7] = ((tmp5)*(tmp5));
REAL_parity_array[8] = tmp5*tmp6;
REAL_parity_array[9] = ((tmp6)*(tmp6));
}

restored runtime settings: ['Infrastructure', 'fp_type']


# Validation Check
### [Back to [top](#Table-of-Contents)]

The generated parity expressions are newly computed. The trusted analytic
cases below are simple enough to check by hand:

- the same point, where all component signs should be `+1`;
- the opposite radial point, where vector signs should be `[-1, +1, -1]`.

The next helper cell parses the generated assignment block. Skim the parser;
the important evidence appears in the following validation cell.

In [4]:
def parse_parity_assignments(c_code):
    symbols = {
        name: sp.symbols(name, real=True)
        for name in ["xx1", "xx2", "xx1_inbounds", "xx2_inbounds"]
    }
    local_dict = {"sin": sp.sin, "cos": sp.cos, **symbols}
    generated_parities = {}
    for raw_line in c_code.splitlines():
        line = raw_line.strip().rstrip(";")
        if not line or "=" not in line:
            continue
        lhs, rhs = [part.strip() for part in line.split("=", 1)]
        if lhs.startswith("const REAL "):
            name = lhs.replace("const REAL ", "").strip()
            local_dict[name] = sp.sympify(rhs, locals=local_dict)
        elif lhs.startswith("REAL_parity_array["):
            index = int(lhs.split("[", 1)[1].split("]", 1)[0])
            generated_parities[index] = sp.sympify(rhs, locals=local_dict)
    return symbols, generated_parities

The validation cell prints the complete sign lists for both trusted cases,
then raises an error if any sign differs from the expected result.

In [5]:
symbols, generated_parities = parse_parity_assignments(assignment_block)
missing_parities = sorted(set(range(10)).difference(generated_parities))
if missing_parities:
    raise RuntimeError(f"Missing parity assignments: {missing_parities}")

theta = sp.pi / 3
phi = sp.pi / 5
same_point = {
    symbols["xx1"]: theta,
    symbols["xx2"]: phi,
    symbols["xx1_inbounds"]: theta,
    symbols["xx2_inbounds"]: phi,
}
opposite_radial_point = {
    symbols["xx1"]: theta,
    symbols["xx2"]: phi,
    symbols["xx1_inbounds"]: sp.pi - theta,
    symbols["xx2_inbounds"]: phi + sp.pi,
}
same_point_signs = [
    sp.simplify(generated_parities[index].subs(same_point)) for index in range(10)
]
opposite_radial_signs = [
    sp.simplify(generated_parities[index].subs(opposite_radial_point))
    for index in range(10)
]
expected_same_point_signs = [1] * 10
expected_opposite_radial_signs = [1, -1, 1, -1, 1, -1, 1, 1, -1, 1]
print("same-point signs:", same_point_signs)
print("opposite-radial signs:", opposite_radial_signs)
if same_point_signs != expected_same_point_signs:
    raise RuntimeError("Unexpected same-point parity signs.")
if opposite_radial_signs != expected_opposite_radial_signs:
    raise RuntimeError("Unexpected opposite-radial parity signs.")
print("validation status: parity signs passed")

same-point signs: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
opposite-radial signs: [1, -1, 1, -1, 1, -1, 1, 1, -1, 1]
validation status: parity signs passed


# Step 4: Check the Outer Radiation Stencil
### [Back to [top](#Table-of-Contents)]

The outer boundary uses a Sommerfeld-like outgoing-wave idea: waves should
leave the domain instead of entering from the edge. The fourth-order one-sided
radial derivative used here has stencil width five, with offsets
`[-3, -2, -1, 0, 1]`.

The trusted coefficients are

$$
\left[-\frac{1}{12}, \frac{1}{2}, -\frac{3}{2}, \frac{5}{6}, \frac{1}{4}\right].
$$

Each row below gives one stencil weight and its grid-point offset. The moment
residuals validate that the stencil differentiates `x` correctly and gives
zero for `1`, `x^2`, `x^3`, and `x^4` at the origin.

In [6]:
coeffs, offsets = outer_bcs.get_arb_offset_FD_coeffs_indices(4, -1, 1)
expected_coeffs = [
    sp.Rational(-1, 12),
    sp.Rational(1, 2),
    sp.Rational(-3, 2),
    sp.Rational(5, 6),
    sp.Rational(1, 4),
]
expected_offsets = [-3, -2, -1, 0, 1]
print("coefficient | offset")
for coeff, offset in zip(coeffs, offsets):
    print(coeff, "|", offset)
if list(coeffs) != expected_coeffs or list(offsets) != expected_offsets:
    raise RuntimeError("Unexpected fourth-order offset stencil.")

moment_residuals = []
for power in range(5):
    moment = sum(coeff * offset**power for coeff, offset in zip(coeffs, offsets))
    expected = 1 if power == 1 else 0
    moment_residuals.append(sp.simplify(moment - expected))
print("stencil moment residuals:", moment_residuals)
if moment_residuals != [0, 0, 0, 0, 0]:
    raise RuntimeError("Unexpected stencil moment residuals.")
print("validation status: one-sided stencil moments passed")

coefficient | offset
-1/12 | -3
1/2 | -2
-3/2 | -1
5/6 | 0
1/4 | 1
stencil moment residuals: [0, 0, 0, 0, 0]
validation status: one-sided stencil moments passed


# Step 5: Inspect Generated Boundary Routines
### [Back to [top](#Table-of-Contents)]

The generated functions are registered in a separate Python process so this
notebook does not leak global NRPy registration state.

Inspect the complete generated-function catalog:

| Generated function | Boundary role | What to inspect |
| --- | --- | --- |
| `bcstruct_set_up__rfm__Spherical` | builds boundary metadata | parameters |
| `apply_bcs_outerradiation_and_inner__rfm__Spherical` | outer and inner rules | wrapper |
| `apply_bcs_inner_only` | fills inner points only | `gfs` parameter |
| `apply_bcs_inner_only_specific_gfs` | fills selected fields | parity input |
| `apply_bcs_outerextrap_and_inner` | extrapolated outer rules | body exists |

The first code cell defines the subprocess script. Skim this helper
scaffolding; it exists only to isolate global NRPy registration state. The
following cell runs it and prints the required evidence.

In [7]:
registration_script = r"""
import nrpy.c_function as cfc
import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import register_all

par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fp_type", "double")
cfc.CFunction_dict.clear()
register_all.register_C_functions({"Spherical"}, radiation_BC_fd_order=4)
expected = [
    "apply_bcs_inner_only",
    "apply_bcs_inner_only_specific_gfs",
    "apply_bcs_outerextrap_and_inner",
    "apply_bcs_outerradiation_and_inner__rfm__Spherical",
    "bcstruct_set_up__rfm__Spherical",
]
registered = set(cfc.CFunction_dict)
missing = sorted(set(expected) - registered)
extra = sorted(registered - set(expected))
print("expected functions:", expected)
print("missing functions:", missing)
print("extra functions:", extra)
if missing or extra:
    raise SystemExit("generated function set did not match expectation")
for name in expected:
    cfunc = cfc.CFunction_dict[name]
    print("generated function:", name)
    print("  return type:", cfunc.cfunc_type)
    print("  includes:", cfunc.includes)
    print("  coordinate wrapper:", cfunc.CoordSystem_for_wrapper_func or "None")
    print("  include_CodeParameters_h:", cfunc.include_CodeParameters_h)
    print("  body length:", len(cfunc.body))
    print("  parameter list:")
    print(cfunc.params)
    print("  complete prototype:")
    print(cfunc.function_prototype)
print("validation status: generated boundary routine registration passed")
"""

The output below prints the expected function set, missing and extra names,
complete metadata, and complete prototypes. This is the generated-code
validation evidence for the boundary-routine registry.

In [8]:
import subprocess
import sys
import textwrap

result = subprocess.run(
    [sys.executable, "-c", textwrap.dedent(registration_script)],
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Generated boundary-function registration failed.")

Setting up reference_metric[Spherical]...
expected functions: ['apply_bcs_inner_only', 'apply_bcs_inner_only_specific_gfs', 'apply_bcs_outerextrap_and_inner', 'apply_bcs_outerradiation_and_inner__rfm__Spherical', 'bcstruct_set_up__rfm__Spherical']
missing functions: []
extra functions: []
generated function: apply_bcs_inner_only
  return type: void
  includes: ['BHaH_defines.h']
  coordinate wrapper: None
  include_CodeParameters_h: False
  body length: 329
  parameter list:
const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs
  complete prototype:
void apply_bcs_inner_only(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs);
generated function: apply_bcs_inner_only_specific_gfs
  return type: void
  includes: ['BHaH_defines.h', 'intrinsics/simd_intrinsics.h']
  coordinate wrapper: None
  include_CodeParameters_h: 

The parity sign checks validate the inner-boundary part of the roadmap. The
stencil moments validate the one-sided derivative used by the outer radiation
rule. The generated CFunction metadata validates that BHaH registration
produced complete entry points for boundary metadata, inner fills, outer
radiation fills, and outer extrapolation fills.

# Learning Check
### [Back to [top](#Table-of-Contents)]

Trace one scalar and one vector component through the boundary-fill roadmap:
identify the in-domain source point, the parity sign, and the generated
function that would apply the rule.

# Continue
### [Back to [top](#Table-of-Contents)]

- [Boundary Conditions and Convergence][bc-convergence]
- [Curvilinear Wave Equation][curvi-wave]
- [GeneralRFM and Fisheye Coordinates](generalrfm_and_fisheye.ipynb)
- [Multicoordinate Wave Project][multi-wave]

[bc-convergence]: ../2-numerical_methods/boundary_conditions_and_convergence.ipynb
[curvi-wave]: ../3-wave_equation/wave_equation_curvilinear.ipynb
[multi-wave]: ../3-wave_equation/wave_equation_multicoordinates.ipynb